# 🧬 SPS Self-Specialization — Capability Handler + Hierarchy Demo

**Research claim demonstrated:** a general capability can manage specialized children; an existing capability can reproduce itself as a transient `S0-C` copy; the copy can be transformed by an external AI model; the generated capability can be verified, activated as `S1`, persisted, reloaded, and reused.

This notebook follows the current `main` branch implementation and explicitly demonstrates the **Capability Handler** and the corrected capability hierarchy.

## 🔬 Complete research flow

```text
                    SerializeCapability [S0]
                              │
                 ┌────────────┴────────────┐
                 │                         │
                 ▼                         ▼
       IntegerMultiplication [S1]   FloatMultiplication [S1]
           statically defined          generated at runtime

Missing float request:

SerializeCapability [S0]
        ↓
detect missing [float, float] → float
        ↓
IntegerMultiplication [S1]  ← source capability
        ↓
REPLICATE → transient copy [S0-C]
        ↓
SPECIALIZE → Ollama + Qwen Coder
        ↓
GENERATED → FloatMultiplication
        ↓
VERIFY → syntax/policy + functional cases
        ↓
ACTIVATE → FloatMultiplication [S1]
        ↓
link directly to SerializeCapability [S0]
        ↓
PERSIST → RELOAD → REUSE without another AI call
```

## 1. Install and start Ollama first

**Run this cell first.** Ollama is started from `/content`, a stable directory that is not deleted when the repository is recloned. This avoids the previous `llama-server process has terminated / cannot get current path` failure.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!pkill -9 ollama || true
!pkill -9 llama-server || true
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

# Pull the local coding model used by the prototype.
!ollama pull qwen2.5-coder:7b

## 2. Clone the latest `main` branch and install dependencies

The notebook deliberately performs a fresh clone so an older Colab checkout cannot shadow the implementation.

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest
!echo 'Repository commit:'
!git rev-parse HEAD
!echo '\nAvailable Ollama models:'
!ollama list

## 3. Verify the implementation before running the real model

The deterministic test suite validates the handler, `S0-C` replication, specialization, verification, general-parent hierarchy, persistence, reload, and failure diagnostics. It does not require Ollama.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 4. Run the real SPS experiment

The demo creates a fresh persistent registry under `/tmp/sps-capability-registry` for every run. The first float request therefore starts from a missing capability.

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 5. What the demo must prove

### Capability hierarchy

The **persistent** hierarchy must be:

```text
                    SerializeCapability [S0]
                              │
                 ┌────────────┴────────────┐
                 │                         │
                 ▼                         ▼
       IntegerMultiplication [S1]   FloatMultiplication [S1]
```

It must **not** permanently become `IntegerMultiplication → IntegerMultiplication-child → FloatMultiplication`. The `S0-C` copy is only a transient evolution artifact.

### Capability Handler

Each capability owns a lightweight runtime handler containing:

- capability identity
- executable function
- runtime status
- source code resource
- input contract
- output contract

The handler is the prototype's answer to: **when a capability is created, where is its runtime handling and what resources does it own?**

### State transition

```text
S0 → S0-C → GENERATED → S1
             │
             └── FAILED if generation/verification fails
```

## 6. Supervisor checklist

Look for these sections in the demo output:

- 🔵 **STATE 0** — `SerializeCapability` exists and `IntegerMultiplication` is already an active child.
- 🟡 **NEW REQUEST** — `[float, float] → float` is missing.
- 🔁 **REPLICATION** — `IntegerMultiplication` produces a transient `S0-C` copy.
- 🧠 **SPECIALIZATION** — Ollama + `qwen2.5-coder:7b` transforms the copy into the float implementation.
- 🛡️ **VERIFICATION** — generated source is checked before activation.
- 🟢 **STATE 1** — `FloatMultiplication` becomes active.
- 🧩 **HANDLER** — capability identity, status, resources and execution information are visible.
- 🌳 **HIERARCHY** — `SerializeCapability` has two specialized children: integer and float multiplication.
- 📜 **EVENT TRACE** — replication, specialization, generation, verification, activation and child-link events are visible.
- 💾 **PERSISTENCE** — JSON metadata and Python source are written to the registry.
- 🔄 **RELOAD + REUSE** — the S1 float capability is reconstructed and reused without another AI generation step.

## 7. Persistent research artifacts

```text
/tmp/sps-capability-registry/
├── registry.json                 ← registry index
├── records/<capability-id>.json ← capability metadata/events
└── sources/<id>_<name>.py       ← exact executable capability source
```

The important research result is not simply that Qwen writes Python. The observable sequence is:

**capability gap detection → replication → specialization → verification → activation → hierarchy registration → persistence → reload → reuse**.

## 8. If Ollama fails

Run the diagnostic cell below. It checks the working directory, Ollama processes, API response, and server log.

In [ ]:
%cd /content
!pwd
!echo '\nOllama processes:'
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!echo '\nOllama API:'
!curl -s http://127.0.0.1:11434/api/tags || true
!echo '\nOllama log:'
!cat /tmp/ollama.log